# 2. Climate — tải dữ liệu và khám phá

## Mục tiêu

Tải [Climate-Explorer.zip từ Google Drive](https://drive.google.com/file/d/1yPxTytgYTcc0V9X5w-zk1_SoBts_bZhI/view?usp=drive_link),
kiểm tra snapshot, rồi khám phá độ phủ, dữ liệu thiếu, mùa vụ và kịch bản khí hậu.
Notebook chạy độc lập, không cần PostgreSQL hoặc secrets.

Archive chứa ba bảng chính: quan trắc ngày tại trạm ACIS, khí hậu tháng/năm cấp county,
và mapping business Yelp → county/trạm. Giữ nguyên đường dẫn để các lệnh import vào DB sử dụng.
Ngày bắt đầu/kết thúc, số dòng và kết luận được đọc hoặc tính từ dữ liệu đã tải.


## Chuẩn bị

Chạy từ repository root:

```sh
uv sync --frozen --group notebooks
uv run --frozen --group notebooks jupyter lab --notebook-dir=.
```

Chọn kernel `.venv/bin/python`, rồi **Restart Kernel and Run All**.
Archive khoảng 259 MB; cần khoảng 550 MB cho archive và snapshot giải nén, cộng các output.
Nếu Drive giới hạn tải tự động, tải thủ công và đặt file tại `ARCHIVE_PATH`.

### Giả định và giới hạn

- Station là quan trắc tại điểm; county là trung bình vùng, không phải weather tại từng shop.
- Quan trắc Livneh 1950–2013, mô hình lịch sử 1950–2005 và projections 2006–2099 được giữ riêng.
  Projections không điền vào lịch sử thực đo và không dự báo thời tiết cho một ngày cụ thể.
- Biểu đồ đổi °F → °C và inch → mm. Ngày trạm theo cửa sổ quan trắc địa phương, không tự coi là UTC.
- Missing giữ nguyên; trace `T` được xấp xỉ 0 mm với marker giữ lại. Cờ cộng dồn `A` bị loại khỏi daily summary.
- Chỉ tổng hợp tháng đủ ngày và giá trị hợp lệ; năm cuối snapshot có thể chưa đủ năm.
- Mỗi county có một trọng số khi mô tả phân phối khí hậu. Không join weather với check-in trong notebook này.

Nguồn gốc và đơn vị: [hướng dẫn dataset](../data/README.md#climate-snapshot),
[Climate Explorer](https://crt-climate-explorer.nemac.org/about/),
[tài liệu ACIS](https://docs.rcc-acis.org/acisws/).


In [ ]:
import hashlib
import json
import os
import platform
import shutil
import stat
import tempfile
import zipfile
from datetime import UTC, datetime
from importlib.metadata import version
from pathlib import Path, PurePosixPath

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "sitesense-mpl-cache"))

import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Mở notebook trong repository SiteSense.")
DATA = ROOT / "data/raw/climate"
ARCHIVE_PATH = ROOT / "data/downloads/climate.zip"
DRIVE_URL = "https://drive.google.com/file/d/1yPxTytgYTcc0V9X5w-zk1_SoBts_bZhI/view?usp=drive_link"
EXPECTED_ARCHIVE_BYTES = 259_170_979
EXPECTED_ARCHIVE_SHA256 = "f7043ec2d668184061055b206fc1433be8e3c57c9441c1db930258dcdbfc9a77"
OUTPUT = ROOT / "data/processed/climate"
OUTPUT.mkdir(parents=True, exist_ok=True)
PILOT_COUNTIES = {"42101": "Philadelphia", "12057": "Tampa", "47037": "Nashville"}
PILOT_CITIES = [("Philadelphia", "PA"), ("Tampa", "FL"), ("Nashville", "TN")]
BASELINE = (1976, 2005)
FUTURE_WINDOWS = {"2031–2060": (2031, 2060), "2070–2099": (2070, 2099)}
VARIABLES = ["tmax", "tmin", "pcpn"]
LABELS = {
    "tmax": "TB nhiệt độ cao nhất ngày (°C)",
    "tmin": "TB nhiệt độ thấp nhất ngày (°C)",
    "pcpn": "Tổng lượng mưa (mm)",
}
CITY_COLORS = {"Philadelphia": "#245f93", "Tampa": "#b87924", "Nashville": "#77823e"}
CITY_LINESTYLES = {"Philadelphia": "-", "Tampa": "--", "Nashville": ":"}
CITY_MARKERS = {"Philadelphia": "o", "Tampa": "s", "Nashville": "^"}
SCENARIO_STYLES = {"rcp45": ("#245f93", "-", "RCP4.5"), "rcp85": ("#b87924", "--", "RCP8.5")}
FIGURES, TABLES = [], []
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 15)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "font.family": "DejaVu Sans",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.18,
        "axes.axisbelow": True,
    }
)


def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def save_table(frame, name):
    frame.to_csv(OUTPUT / name, index=False)
    TABLES.append(name)


def show_figure(figure, name):
    figure.savefig(OUTPUT / name, dpi=160, bbox_inches="tight")
    FIGURES.append(name)
    plt.show()
    plt.close(figure)


print(
    {
        "python": platform.python_version(),
        **{name: version(name) for name in ("pandas", "numpy", "matplotlib")},
    }
)
print("Inputs:", DATA.relative_to(ROOT), "| Outputs:", OUTPUT.relative_to(ROOT))

## Các bước

### 1. Tải và xác minh archive

`gdown` tải từ link Drive và tiếp tục file `.part` sau gián đoạn. Archive đã có và đúng checksum
được dùng lại. Với một phiên bản dataset khác, cập nhật các giá trị expected sau khi xác minh nguồn.


In [ ]:
# Archive được ghim theo checksum của snapshot đã kiểm tra.
ARCHIVE_PATH.parent.mkdir(parents=True, exist_ok=True)
if not ARCHIVE_PATH.exists():
    partial_bytes = sum(
        path.stat().st_size for path in ARCHIVE_PATH.parent.glob(f"{ARCHIVE_PATH.name}*.part")
    )
    required_bytes = max(0, EXPECTED_ARCHIVE_BYTES - partial_bytes) + 300 * 1024**2
    if shutil.disk_usage(ARCHIVE_PATH.parent).free < required_bytes:
        raise OSError("Không đủ dung lượng để tải và giải nén snapshot Climate.")
    result = gdown.download(url=DRIVE_URL, output=str(ARCHIVE_PATH), fuzzy=True, resume=True)
    if result is None or not ARCHIVE_PATH.is_file():
        raise RuntimeError("Drive chưa tải được file. Tải thủ công đến ARCHIVE_PATH rồi chạy lại.")
assert ARCHIVE_PATH.stat().st_size == EXPECTED_ARCHIVE_BYTES, "Kích thước archive không khớp."
archive_hash = sha256(ARCHIVE_PATH)
assert archive_hash == EXPECTED_ARCHIVE_SHA256, "Checksum archive không khớp snapshot."
assert zipfile.is_zipfile(ARCHIVE_PATH), "File tải về không phải ZIP."
print(
    {
        "archive": str(ARCHIVE_PATH.relative_to(ROOT)),
        "bytes": ARCHIVE_PATH.stat().st_size,
        "sha256": archive_hash,
    }
)

### 2. Giải nén snapshot vào đường dẫn chuẩn

Kiểm tra CRC, đường dẫn và checksum trước khi sử dụng. Archive chỉ được giải nén dưới
`data/raw/climate/`; không ghi đè một snapshot local khác. Bảng mapping đã có sẵn,
không cần gọi API ACIS, dựng lại county/station mapping hoặc tái tạo ERA5.


In [ ]:
# Original member root inside the pinned ZIP; local DATA uses the normalized path.
PREFIX = PurePosixPath("data/raw/climate_explorer")
REQUIRED_FILES = {
    "acquisition.json",
    "yelp_counties/climate.csv.gz",
    "yelp_counties/manifest.json",
    "yelp_stations/observations.csv.gz",
    "yelp_stations/manifest.json",
    "yelp_scope/business_climate_mapping.csv",
    "yelp_scope/scope_manifest.json",
}


def validate_snapshot(directory):
    for folder in ("yelp_counties", "yelp_stations"):
        manifest = json.loads((directory / folder / "manifest.json").read_text())
        assert manifest["status"] == "complete", folder
        filename = "climate.csv.gz" if folder == "yelp_counties" else "observations.csv.gz"
        assert sha256(directory / folder / filename) == manifest["csv"]["sha256"], folder
    scope = json.loads((directory / "yelp_scope/scope_manifest.json").read_text())
    mapping = next(
        item for item in scope["outputs"] if item["path"].endswith("business_climate_mapping.csv")
    )
    assert sha256(directory / "yelp_scope/business_climate_mapping.csv") == mapping["sha256"]
    assert json.loads((directory / "acquisition.json").read_text())["status"] == "complete"


with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    entries = {}
    for member in archive.infolist():
        path = PurePosixPath(member.filename)
        if (
            path.is_absolute()
            or ".." in path.parts
            or "\\" in member.filename
            or stat.S_ISLNK(member.external_attr >> 16)
        ):
            raise ValueError(f"Archive member không an toàn: {member.filename}")
        relative = path.relative_to(PREFIX)  # Từ chối file ngoài root đã biết.
        if not member.is_dir():
            assert str(relative) not in entries, "Archive có đường dẫn trùng lặp."
            entries[str(relative)] = member
    assert REQUIRED_FILES <= entries.keys(), "Archive thiếu input cần cho app."
    assert archive.testzip() is None, "Archive chứa file bị lỗi CRC."
    source_hashes = {}
    for relative, member in entries.items():
        with archive.open(member) as stream:
            source_hashes[relative] = hashlib.file_digest(stream, "sha256").hexdigest()
    local_matches = DATA.is_dir() and all(
        (DATA / relative).is_file()
        and not (DATA / relative).is_symlink()
        and sha256(DATA / relative) == digest
        for relative, digest in source_hashes.items()
    )
    if local_matches:
        print("Dùng lại snapshot local: toàn bộ file khớp archive Drive.")
    else:
        if DATA.exists() and any(DATA.iterdir()):
            raise ValueError(
                "Snapshot local khác archive. Kiểm tra/di chuyển DATA trước khi giải nén."
            )
        DATA.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix="climate-stage-", dir=DATA.parent) as temporary:
            staged = Path(temporary) / "climate_explorer"
            staged.mkdir()
            for relative, member in entries.items():
                destination = staged / relative
                destination.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, destination.open("wb") as target:
                    shutil.copyfileobj(source, target)
                assert sha256(destination) == source_hashes[relative]
            validate_snapshot(staged)
            if DATA.exists():
                DATA.rmdir()  # Chỉ thay thư mục trống.
            staged.rename(DATA)
        print("Đã giải nén và kiểm tra snapshot Climate.")
validate_snapshot(DATA)
download_manifest = {
    "status": "complete",
    "verified_at_utc": datetime.now(UTC).isoformat(),
    "source_url": DRIVE_URL,
    "archive_sha256": archive_hash,
    "archive_bytes": ARCHIVE_PATH.stat().st_size,
    "files": len(entries),
    "extracted_bytes": sum(member.file_size for member in entries.values()),
    "snapshot": str(DATA.relative_to(ROOT)),
    "file_hashes": source_hashes,
}
(ARCHIVE_PATH.parent / "climate.manifest.json").write_text(
    json.dumps(download_manifest, indent=2) + "\n"
)
print({key: download_manifest[key] for key in ("files", "extracted_bytes", "snapshot")})

### 3. Kiểm tra manifest và checksum dữ liệu

Đối chiếu hash CSV với manifest nguồn và mapping. `complete` nghĩa là tải đủ snapshot,
không có nghĩa các quan trắc đều không thiếu.

In [ ]:
acquisition = json.loads((DATA / "acquisition.json").read_text())
county_manifest = json.loads((DATA / "yelp_counties/manifest.json").read_text())
station_manifest = json.loads((DATA / "yelp_stations/manifest.json").read_text())
scope_manifest = json.loads((DATA / "yelp_scope/scope_manifest.json").read_text())
assert all(
    item["status"] == "complete" for item in (acquisition, county_manifest, station_manifest)
)
INPUT_HASHES = {}
for relative, manifest in [
    ("yelp_counties/climate.csv.gz", county_manifest),
    ("yelp_stations/observations.csv.gz", station_manifest),
]:
    INPUT_HASHES[relative] = sha256(DATA / relative)
    assert INPUT_HASHES[relative] == manifest["csv"]["sha256"], relative
for item in scope_manifest["outputs"]:
    if item["path"].endswith("business_climate_mapping.csv"):
        relative = "yelp_scope/business_climate_mapping.csv"
        INPUT_HASHES[relative] = sha256(DATA / relative)
        assert INPUT_HASHES[relative] == item["sha256"]
for name in [
    "acquisition.json",
    "yelp_counties/manifest.json",
    "yelp_stations/manifest.json",
    "yelp_scope/scope_manifest.json",
]:
    INPUT_HASHES[name] = sha256(DATA / name)
source_audit = pd.DataFrame(
    [
        {
            "dataset": item["dataset"],
            "rows": item["rows"],
            "missing": item["missing_values"],
            "sha256": item["sha256"],
        }
        for item in acquisition["artifacts"]
    ]
)
display(source_audit)
print("Snapshot complete (UTC):", acquisition["completed_at_utc"])

### 4. Đọc ba bảng và giữ đúng đơn vị phân tích

ID county/station/network luôn là chuỗi; đặc biệt FIPS có số 0 ở đầu.
Bảng county có hai frequency và nhiều statistic, cần lọc trước khi tính trung bình.
Bảng station giữ nguyên `raw_value`, flag, nguồn và observation time.

In [ ]:
counties = pd.read_csv(
    DATA / "yelp_counties/climate.csv.gz", dtype={"county_fips": str, "period": str}
)
stations = pd.read_csv(
    DATA / "yelp_stations/observations.csv.gz",
    keep_default_na=False,
    dtype={
        "station_id": str,
        "station_county_fips": str,
        "network_id": str,
        "source_flag": str,
        "observation_time_local_standard": str,
        "raw_value": str,
        "flag": str,
        "unit": str,
    },
)
stations["value"] = pd.to_numeric(stations["value"].replace("", np.nan), errors="raise")
stations["date"] = pd.to_datetime(stations["date"], format="%Y-%m-%d")
mapping = pd.read_csv(
    DATA / "yelp_scope/business_climate_mapping.csv",
    dtype={"county_fips": str, "station_id": str},
    keep_default_na=False,
)
stations["year"] = stations["date"].dt.year
stations["flag_clean"] = stations["flag"].str.strip()
for frame, columns in [
    (
        counties,
        [
            "county_fips",
            "dataset",
            "scenario",
            "statistic",
            "frequency",
            "variable",
            "unit",
            "grid",
        ],
    ),
    (stations, ["station_id", "variable", "unit", "flag_clean"]),
]:
    for column in columns:
        frame[column] = frame[column].astype("category")
for title, frame in [("County", counties), ("Station", stations), ("Business mapping", mapping)]:
    print(title, frame.shape)
    display(frame.head(3))

### 5. Kiểm tra grain, completeness và ràng buộc cơ bản

Các assert kiểm tra uniqueness, phạm vi ID, số ngày cho từng station/variable, số dòng
annual/monthly của từng nhóm county, giá trị hữu hạn và đơn vị. Kiểm tra nhiệt độ
cao nhất thấp hơn nhiệt độ thấp nhất và mưa âm được báo riêng, không âm thầm sửa dữ liệu.

In [ ]:
station_keys = ["station_id", "date", "variable"]
county_keys = ["county_fips", "frequency", "dataset", "scenario", "statistic", "period", "variable"]
assert not mapping["business_id"].duplicated().any()
assert not stations.duplicated(station_keys).any()
assert not counties.duplicated(county_keys).any()
assert len(mapping) == scope_manifest["businesses"]
assert len(counties) == county_manifest["rows"] and len(stations) == station_manifest["rows"]
assert int(stations["value"].isna().sum()) == station_manifest["missing_values"]
assert int(counties["value"].isna().sum()) == county_manifest["missing_values"]
mapped = mapping.loc[mapping["status"].eq("mapped")].copy()
assert set(mapped["county_fips"]) == set(counties["county_fips"])
assert set(mapped["station_id"]) == set(stations["station_id"])
assert set(stations["variable"]) == set(counties["variable"]) == set(VARIABLES)
requested_dates = pd.date_range(
    station_manifest["scope"]["start"], station_manifest["scope"]["end"]
)
assert stations["date"].min() == requested_dates.min()
assert stations["date"].max() == requested_dates.max()
station_sizes = stations.groupby(["station_id", "variable"], observed=True).size()
assert station_sizes.eq(len(requested_dates)).all()
expected_years = {
    "observations": (1950, 2013),
    "historical_modeled": (1950, 2005),
    "projections": (2006, 2099),
}
for keys, group in counties.groupby(county_keys[:5] + ["variable"], observed=True):
    _, frequency, dataset, _, _, _ = keys
    start, end = expected_years[dataset]
    expected = (
        [str(year) for year in range(start, end + 1)]
        if frequency == "annual"
        else pd.period_range(f"{start}-01", f"{end}-12", freq="M").astype(str).tolist()
    )
    assert set(group["period"]) == set(expected) and len(group) == len(expected), keys
for frame in [stations, counties]:
    assert np.isfinite(frame["value"].dropna()).all()
    for variable, unit in [("tmax", "degreeF"), ("tmin", "degreeF"), ("pcpn", "inch")]:
        assert set(frame.loc[frame["variable"].eq(variable), "unit"]) == {unit}
wide_temperature = stations.loc[stations["variable"].isin(["tmax", "tmin"])].pivot(
    index=["station_id", "date"], columns="variable", values="value"
)
invalid_temperature_pairs = int(wide_temperature["tmax"].lt(wide_temperature["tmin"]).sum())
negative_precipitation = int(stations.loc[stations["variable"].eq("pcpn"), "value"].lt(0).sum())
checks = pd.DataFrame(
    {
        "check": [
            "Duplicate business / station / county keys",
            "Finite numeric values",
            "Station ID/date/variable completeness",
            "County group period completeness",
            "tmax < tmin",
            "Negative precipitation",
        ],
        "result": [
            "PASS",
            "PASS",
            "PASS",
            "PASS",
            invalid_temperature_pairs,
            negative_precipitation,
        ],
    }
)
display(checks)
save_table(checks, "data_checks.csv")

### 6. Độ phủ địa điểm và khoảng cách tới trạm

Địa điểm Canada nằm ngoài nguồn county Mỹ và được giữ với status riêng. Đếm business
chỉ mô tả phạm vi Yelp, không phải trọng số diện tích hoặc dân số. State từ tọa độ có
thể khác nhãn nguồn; notebook báo các mismatch này thay vì dùng nhãn nguồn để sửa tọa độ.

In [ ]:
scope_counts = mapping.groupby("status").size().rename("businesses").reset_index()
scope_counts["share_pct"] = scope_counts["businesses"] / len(mapping) * 100
county_businesses = (
    mapped.groupby(["county_fips", "county_name", "county_state"])
    .size()
    .rename("businesses")
    .reset_index()
    .sort_values("businesses", ascending=False)
)
station_distance = pd.to_numeric(mapped["station_distance_km"], errors="raise")
state_mismatches = mapping["state_mismatch"].astype(str).str.lower().eq("true").sum()
display(scope_counts)
display(county_businesses.head(12))
display(station_distance.describe(percentiles=[0.5, 0.95]).to_frame("distance_km"))
print(
    "County / station:",
    mapped["county_fips"].nunique(),
    "/",
    mapped["station_id"].nunique(),
    "| Source-state mismatches:",
    int(state_mismatches),
)
save_table(scope_counts, "scope_counts.csv")
save_table(county_businesses, "county_business_counts.csv")
fig, axes = plt.subplots(1, 2, figsize=(13, 5), layout="constrained")
top = county_businesses.head(12).iloc[::-1]
axes[0].barh(top["county_name"] + ", " + top["county_state"], top["businesses"], color="#245f93")
axes[0].set(title="12 county có nhiều địa điểm Yelp nhất", xlabel="Số business (snapshot Yelp)")
axes[1].hist(station_distance, bins=np.arange(0, 66, 3), color="#245f93", edgecolor="white")
for quantile, style in [(0.5, "-"), (0.95, "--")]:
    distance = station_distance.quantile(quantile)
    axes[1].axvline(
        distance,
        color="#43484c",
        linestyle=style,
        label=f"P{int(quantile * 100)} = {distance:.1f} km",
    )
axes[1].set(
    title="Khoảng cách business → trạm được chọn", xlabel="Khoảng cách (km)", ylabel="Số business"
)
axes[1].legend()
fig.suptitle(f"Phạm vi Mỹ: {len(mapped):,} business; mỗi business xuất hiện một lần", fontsize=13)
show_figure(fig, "01_scope_and_station_distance.png")

### 7. Dữ liệu thiếu và các cờ nguồn

Phân biệt **numeric available** và **usable cho daily summary**. Policy bảo thủ:
chỉ dùng numeric với flag trống hoặc trace `T`; loại `A`, `S`, `M` và flag khác.
`S` được giữ nguyên như cờ nguồn, không tự gán nghĩa “suspect”. Nguồn ACIS có thể
merge network; bảng network phía dưới cho thấy dữ liệu không hoàn toàn chỉ từ một mạng.

In [ ]:
stations["numeric_available"] = stations["value"].notna()
stations["usable"] = stations["numeric_available"] & stations["flag_clean"].isin(["", "T"])
stations["metric_value"] = np.where(
    stations["variable"].eq("pcpn"), stations["value"] * 25.4, (stations["value"] - 32) * 5 / 9
)
# Trace represents an unmeasurably small amount; summary uses a stated zero approximation.
stations.loc[stations["variable"].eq("pcpn") & stations["flag_clean"].eq("T"), "metric_value"] = 0.0
stations["usable_value"] = stations["metric_value"].where(stations["usable"])
quality = stations.groupby("variable", observed=True).agg(
    requested=("value", "size"), numeric=("numeric_available", "sum"), usable=("usable", "sum")
)
quality["missing"] = quality["requested"] - quality["numeric"]
quality["missing_pct"] = quality["missing"] / quality["requested"] * 100
quality["not_usable_pct"] = (1 - quality["usable"] / quality["requested"]) * 100
flag_counts = (
    stations.groupby(["variable", "flag_clean"], observed=True).size().rename("rows").reset_index()
)
flag_counts["flag_clean"] = flag_counts["flag_clean"].astype(str).replace("", "(blank)")
network_counts = stations.groupby("network_id", dropna=False).size().rename("rows").reset_index()
display(quality.reset_index())
display(flag_counts)
display(network_counts)
save_table(quality.reset_index(), "station_quality_by_variable.csv")
save_table(flag_counts, "station_flag_counts.csv")
save_table(network_counts, "station_network_counts.csv")

### 8. Missingness theo station × năm và biến

Heatmap dùng số giá trị thiếu chia cho **số ngày đã yêu cầu** của station/variable/năm.
Năm cuối chỉ đến ngày kết thúc request, không dùng 365 ngày làm mẫu số và không
coi các ngày tương lai là missing. Ba panel có cùng thang màu 0–100%.

In [ ]:
annual_coverage = (
    stations.groupby(["station_id", "year", "variable"], observed=True)
    .agg(
        requested_days=("value", "size"),
        numeric_days=("numeric_available", "sum"),
        usable_days=("usable", "sum"),
    )
    .reset_index()
)
annual_coverage["missing_pct"] = (
    1 - annual_coverage["numeric_days"] / annual_coverage["requested_days"]
) * 100
annual_coverage["not_usable_pct"] = (
    1 - annual_coverage["usable_days"] / annual_coverage["requested_days"]
) * 100
station_order = sorted(stations["station_id"].unique())
years = sorted(stations["year"].unique())
year_ticks = sorted(set(range(0, len(years), 2)) | {len(years) - 1})
fig, axes = plt.subplots(1, 3, figsize=(16, 11), sharey=True, layout="constrained")
for axis, variable in zip(axes, VARIABLES, strict=True):
    heat = (
        annual_coverage.loc[annual_coverage["variable"].eq(variable)]
        .pivot(index="station_id", columns="year", values="missing_pct")
        .reindex(index=station_order, columns=years)
    )
    image = axis.imshow(heat, aspect="auto", cmap="Blues", vmin=0, vmax=100)
    axis.set(
        title=variable,
        xlabel=f"Năm ({requested_dates.max().year} là năm cuối snapshot)",
        xticks=year_ticks,
        xticklabels=[years[index] for index in year_ticks],
    )
    axis.tick_params(axis="x", labelrotation=90)
axes[0].set(yticks=np.arange(len(station_order)), yticklabels=station_order, ylabel="Station ID")
axes[0].tick_params(axis="y", labelsize=7)
fig.colorbar(image, ax=axes, shrink=0.8, label="Missing / requested days (%)")
fig.suptitle(
    f"Quan trắc numeric: {station_manifest['scope']['start']} → {station_manifest['scope']['end']}"
)
show_figure(fig, "02_station_missingness_by_year.png")
save_table(annual_coverage, "station_annual_coverage.csv")

### 9. Năm cuối snapshot thực tế đã có dữ liệu đến ngày nào?

Bảng dưới tính lại ngày mới nhất từ từng station/variable, đối chiếu với manifest,
và đếm riêng phần numeric/usable. Các ngày mới nhất là **quan trắc đã có trong snapshot**,
không phải bằng chứng dữ liệu đã được phát hành đúng ngày đó trong lịch sử.

In [ ]:
latest_all = (
    stations.loc[stations["numeric_available"]]
    .groupby(["station_id", "variable"], observed=True)["date"]
    .max()
)
for item in station_manifest["stations"]:
    for variable in VARIABLES:
        date = latest_all.loc[(item["station_id"], variable)].strftime("%Y-%m-%d")
        assert date == item["coverage"][variable]["latest_observed_date"]
latest_year = requested_dates.max().year
latest_slice = stations.loc[stations["year"].eq(latest_year)]
freshness = latest_slice.groupby(["station_id", "variable"], observed=True).agg(
    requested_days=("value", "size"),
    numeric_days=("numeric_available", "sum"),
    usable_days=("usable", "sum"),
)
freshness["latest_numeric_date"] = (
    latest_slice.loc[latest_slice["numeric_available"]]
    .groupby(["station_id", "variable"], observed=True)["date"]
    .max()
)
freshness["latest_usable_date"] = (
    latest_slice.loc[latest_slice["usable"]]
    .groupby(["station_id", "variable"], observed=True)["date"]
    .max()
)
freshness["missing_pct"] = (1 - freshness["numeric_days"] / freshness["requested_days"]) * 100
freshness["lag_days_from_request_end"] = (
    requested_dates.max() - freshness["latest_numeric_date"]
).dt.days
freshness = freshness.reset_index().sort_values(
    ["lag_days_from_request_end", "missing_pct"], ascending=False
)
display(freshness.head(12))
print(
    f"{latest_year}: {latest_slice['date'].min().date()} → {requested_dates.max().date()}, "
    f"{freshness['requested_days'].max()} requested days; chưa đủ năm."
)
print(
    "Latest numeric date range:",
    freshness["latest_numeric_date"].min().date(),
    "→",
    freshness["latest_numeric_date"].max().date(),
)
save_table(freshness, "station_latest_year_freshness.csv")

### 10. Chọn trạm minh họa cho ba thành phố

Chọn trạm được ghép với nhiều business nhất trong từng city/state. Đây là policy minh họa
cố định, không chọn trạm sau khi nhìn xu hướng. Một thành phố vẫn có thể có nhiều trạm;
bảng nêu rõ tỷ lệ business do trạm minh họa phục vụ.

In [ ]:
pilot_rows = []
for city, state in PILOT_CITIES:
    subset = mapped.loc[mapped["city"].eq(city) & mapped["source_state"].eq(state)]
    counts = (
        subset.groupby(["station_id", "station_name"]).size().rename("businesses").reset_index()
    )
    selected = counts.sort_values(["businesses", "station_id"], ascending=[False, True]).iloc[0]
    pilot_rows.append(
        {
            "city": city,
            "state": state,
            "station_id": selected["station_id"],
            "station_name": selected["station_name"],
            "city_businesses": len(subset),
            "selected_station_businesses": int(selected["businesses"]),
            "selected_share_pct": selected["businesses"] / len(subset) * 100,
        }
    )
pilot_stations = pd.DataFrame(pilot_rows)
display(pilot_stations)
save_table(pilot_stations, "pilot_station_selection.csv")

### 11. Tổng hợp tháng — chỉ tháng đủ dữ liệu

Mean cho `tmax/tmin`, sum cho `pcpn`. Một tháng hợp lệ cần đủ số ngày trong tháng,
đủ giá trị theo policy flag, và nằm trọn trong request. Cả mean/sum của tháng chưa đủ
đều bị mask. `pcpn` là tổng reported precipitation xấp xỉ do trace được coi bằng 0.

In [ ]:
stations["month"] = stations["date"].dt.to_period("M").astype(str)
monthly = (
    stations.groupby(["station_id", "month", "variable"], observed=True)
    .agg(
        requested_days=("value", "size"),
        usable_days=("usable", "sum"),
        observed_mean=("usable_value", "mean"),
        reported_sum=("usable_value", lambda values: values.sum(min_count=1)),
    )
    .reset_index()
)
monthly["month_start"] = pd.to_datetime(monthly["month"] + "-01")
monthly["calendar_days"] = monthly["month_start"].dt.days_in_month
monthly["complete"] = monthly["requested_days"].eq(monthly["calendar_days"]) & monthly[
    "usable_days"
].eq(monthly["calendar_days"])
monthly["metric"] = np.where(
    monthly["variable"].eq("pcpn"), monthly["reported_sum"], monthly["observed_mean"]
)
monthly["metric"] = monthly["metric"].where(monthly["complete"])
monthly["unit"] = np.where(monthly["variable"].eq("pcpn"), "mm", "degreeC")
assert monthly.loc[~monthly["complete"], "metric"].isna().all()
assert (
    not monthly.loc[monthly["month"].eq(requested_dates.max().strftime("%Y-%m")), "complete"].any()
    if requested_dates.max().day != requested_dates.max().days_in_month
    else True
)
monthly_summary = (
    monthly.groupby("variable", observed=True)
    .agg(station_months=("complete", "size"), complete_months=("complete", "sum"))
    .reset_index()
)
monthly_summary["excluded_months"] = (
    monthly_summary["station_months"] - monthly_summary["complete_months"]
)
display(monthly_summary)
save_table(
    monthly.drop(columns=["observed_mean", "reported_sum"]),
    "station_complete_monthly_summaries.csv",
)
pilot_monthly = monthly.merge(
    pilot_stations[["station_id", "city"]], on="station_id", validate="many_to_one"
)
pilot_monthly_quality = (
    pilot_monthly.groupby(["city", "variable"], observed=True)
    .agg(requested_months=("complete", "size"), complete_months=("complete", "sum"))
    .reset_index()
)
pilot_monthly_quality["complete_pct"] = (
    pilot_monthly_quality["complete_months"] / pilot_monthly_quality["requested_months"] * 100
)
display(pilot_monthly_quality)
save_table(pilot_monthly_quality, "pilot_complete_month_coverage.csv")
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True, layout="constrained")
for axis, variable in zip(axes, VARIABLES, strict=True):
    for city, _ in PILOT_CITIES:
        series = pilot_monthly.loc[
            pilot_monthly["city"].eq(city) & pilot_monthly["variable"].eq(variable)
        ].sort_values("month_start")
        axis.plot(
            series["month_start"],
            series["metric"],
            color=CITY_COLORS[city],
            linestyle=CITY_LINESTYLES[city],
            label=city,
            linewidth=1,
        )
    axis.set(ylabel=LABELS[variable])
axes[0].legend(ncol=3)
axes[-1].set_xlabel("Tháng; gap = tháng không đủ dữ liệu theo policy")
fig.suptitle(
    f"Quan trắc trạm: {requested_dates.min().year}–{requested_dates.max().year} (tháng đủ dữ liệu)"
)
show_figure(fig, "03_pilot_station_monthly_observations.png")

### 12. Mùa vụ lịch sử ở ba trạm minh họa

Lấy trung bình theo tháng trong năm từ các tháng hoàn chỉnh trước năm cuối snapshot;
không đưa năm cuối đang dở vào baseline. Số tháng thực sự góp vào mỗi điểm được xuất riêng.
Đây là mô tả snapshot lịch sử, không phải climate normal 30 năm chính thức.

In [ ]:
seasonal_input = pilot_monthly.loc[
    pilot_monthly["month_start"].dt.year.lt(latest_year) & pilot_monthly["complete"]
].copy()
seasonal_input["calendar_month"] = seasonal_input["month_start"].dt.month
seasonality = (
    seasonal_input.groupby(["city", "calendar_month", "variable"], observed=True)
    .agg(metric=("metric", "mean"), contributing_months=("metric", "size"))
    .reset_index()
)
display(
    seasonality.groupby(["city", "variable"])["contributing_months"]
    .agg(["min", "max"])
    .reset_index()
)
save_table(seasonality, "pilot_station_seasonality.csv")
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), layout="constrained")
for axis, variable in zip(axes, VARIABLES, strict=True):
    for city, _ in PILOT_CITIES:
        series = seasonality.loc[
            seasonality["city"].eq(city) & seasonality["variable"].eq(variable)
        ]
        axis.plot(
            series["calendar_month"],
            series["metric"],
            marker=CITY_MARKERS[city],
            linestyle=CITY_LINESTYLES[city],
            color=CITY_COLORS[city],
            label=city,
        )
    axis.set(title=variable, xlabel="Tháng trong năm", ylabel=LABELS[variable], xticks=range(1, 13))
    if variable == "pcpn":
        axis.set_ylim(bottom=0)
axes[0].legend()
fig.suptitle("Mùa vụ 2009–2025: chỉ tháng đủ dữ liệu; số tháng góp vào mỗi điểm có thể khác nhau")
show_figure(fig, "04_pilot_station_seasonality.png")

### 13. County lịch sử: quan trắc Livneh và mô hình được giữ riêng

Ở đây chỉ dùng frequency annual và statistic observed/weighted_mean. Không kéo quan
trắc Livneh qua 2013 và không dùng projections 2014–2026 để điền lịch sử quan trắc.
Đường smooth là rolling mean 10 năm, yêu cầu đủ 10 năm; gap vẫn được giữ.

In [ ]:
annual = counties.loc[counties["frequency"].eq("annual")].copy()
annual["year"] = annual["period"].astype(int)
annual["metric"] = np.where(
    annual["variable"].eq("pcpn"), annual["value"] * 25.4, (annual["value"] - 32) * 5 / 9
)
county_inventory = (
    counties.groupby(["frequency", "dataset", "scenario", "statistic"], observed=True)
    .agg(
        county_count=("county_fips", "nunique"),
        rows=("value", "size"),
        first_period=("period", "min"),
        last_period=("period", "max"),
    )
    .reset_index()
)
display(county_inventory)
save_table(county_inventory, "county_dataset_inventory.csv")
fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True, layout="constrained")
for row, (fips, city) in enumerate(PILOT_COUNTIES.items()):
    for col, variable in enumerate(["tmax", "pcpn"]):
        axis = axes[row, col]
        for dataset, statistic, color, style, label in [
            ("observations", "observed", "#245f93", "-", "Livneh quan trắc"),
            ("historical_modeled", "weighted_mean", "#43484c", "--", "LOCA mô hình lịch sử"),
        ]:
            series = annual.loc[
                annual["county_fips"].eq(fips)
                & annual["variable"].eq(variable)
                & annual["dataset"].eq(dataset)
                & annual["statistic"].eq(statistic)
            ].sort_values("year")
            axis.plot(series["year"], series["metric"], color=color, alpha=0.2, linewidth=0.7)
            axis.plot(
                series["year"],
                series["metric"].rolling(10, min_periods=10).mean(),
                color=color,
                linestyle=style,
                label=label,
            )
        axis.set(title=f"{city} county · {variable}", ylabel=LABELS[variable])
        if variable == "pcpn":
            axis.set_ylim(bottom=0)
axes[0, 0].legend(fontsize=9)
for axis in axes[-1]:
    axis.set_xlabel("Năm; đậm = rolling mean 10 năm")
fig.suptitle("County annual data: Livneh 1950–2013 và LOCA modeled history 1950–2005")
show_figure(fig, "05_county_historical_observed_and_modeled.png")

### 14. RCP4.5 và RCP8.5: thay đổi so với cùng mô hình lịch sử

Baseline chung **1976–2005**: trung bình annual weighted_mean của cùng county/variable,
30 năm đủ dữ liệu. Không lấy station/Livneh làm baseline để tránh trộn bias nguồn.

Với nhiệt độ: Δ°C = projected °C − baseline °C. Với mưa: Δ% =
(projected mm / baseline mm − 1) × 100. Đường đậm là rolling mean 10 năm của annual
weighted_mean; vùng mờ là rolling mean của annual ensemble min/max theo cùng baseline.
Vùng mờ không phải CI, percentile hoặc trajectory của một model cụ thể.

In [ ]:
model_history = annual.loc[
    annual["dataset"].eq("historical_modeled")
    & annual["statistic"].eq("weighted_mean")
    & annual["year"].between(*BASELINE)
]
baseline = (
    model_history.groupby(["county_fips", "variable"], observed=True)
    .agg(baseline_metric=("metric", "mean"), baseline_years=("year", "size"))
    .reset_index()
)
assert baseline["baseline_years"].eq(BASELINE[1] - BASELINE[0] + 1).all()
assert baseline.loc[baseline["variable"].eq("pcpn"), "baseline_metric"].gt(0).all()
projections = annual.loc[annual["dataset"].eq("projections")].merge(
    baseline, on=["county_fips", "variable"], validate="many_to_one"
)
projections["change"] = np.where(
    projections["variable"].eq("pcpn"),
    (projections["metric"] / projections["baseline_metric"] - 1) * 100,
    projections["metric"] - projections["baseline_metric"],
)
assert set(projections["scenario"]) == {"rcp45", "rcp85"}
fig, axes = plt.subplots(3, 3, figsize=(15, 10), sharex=True, sharey="col", layout="constrained")
for row, (fips, city) in enumerate(PILOT_COUNTIES.items()):
    for col, variable in enumerate(VARIABLES):
        axis = axes[row, col]
        for scenario, (color, style, label) in SCENARIO_STYLES.items():
            series = (
                projections.loc[
                    projections["county_fips"].eq(fips)
                    & projections["variable"].eq(variable)
                    & projections["scenario"].eq(scenario)
                ]
                .pivot(index="year", columns="statistic", values="change")
                .sort_index()
            )
            smooth = series.rolling(10, min_periods=10).mean()
            axis.fill_between(smooth.index, smooth["min"], smooth["max"], color=color, alpha=0.11)
            axis.plot(
                smooth.index, smooth["weighted_mean"], color=color, linestyle=style, label=label
            )
        axis.axhline(0, color="#43484c", linewidth=0.8)
        axis.set(
            title=f"{city} county · {variable}",
            ylabel="Δ tổng mưa (%)" if variable == "pcpn" else "Δ nhiệt độ (°C)",
        )
axes[0, 0].legend()
for axis in axes[-1]:
    axis.set_xlabel("Năm cuối của rolling window 10 năm")
fig.suptitle(f"Kịch bản khí hậu county 2006–2099 · baseline LOCA {BASELINE[0]}–{BASELINE[1]}")
show_figure(fig, "06_county_projection_changes.png")

### 15. So sánh cửa sổ 30 năm cho toàn bộ 58 county

Bảng dùng weighted_mean của mô hình, **không lấy trung bình min/mean/max như các sample**.
Mỗi county có trọng số bằng nhau trong boxplot. Số business được giữ làm context,
không làm số lượng quan trắc hay trọng số cho phân phối khí hậu.
Mưa là trung bình annual totals trong mỗi cửa sổ, không phải tổng cộng 30 năm.

In [ ]:
window_tables = []
weighted_projection = projections.loc[projections["statistic"].eq("weighted_mean")]
for label, (first, last) in FUTURE_WINDOWS.items():
    period = weighted_projection.loc[weighted_projection["year"].between(first, last)]
    summary = (
        period.groupby(
            ["county_fips", "county_name", "state", "variable", "scenario"], observed=True
        )
        .agg(
            future_metric=("metric", "mean"),
            baseline_metric=("baseline_metric", "first"),
            years=("year", "size"),
        )
        .reset_index()
    )
    assert summary["years"].eq(last - first + 1).all()
    summary["window"] = label
    summary["change"] = np.where(
        summary["variable"].eq("pcpn"),
        (summary["future_metric"] / summary["baseline_metric"] - 1) * 100,
        summary["future_metric"] - summary["baseline_metric"],
    )
    summary["change_unit"] = np.where(summary["variable"].eq("pcpn"), "percent", "degreeC")
    window_tables.append(summary)
window_changes = pd.concat(window_tables, ignore_index=True).merge(
    county_businesses[["county_fips", "businesses"]], on="county_fips", validate="many_to_one"
)
assert len(window_changes) == mapped["county_fips"].nunique() * len(VARIABLES) * 2 * len(
    FUTURE_WINDOWS
)
pilot_changes = window_changes.loc[window_changes["county_fips"].isin(PILOT_COUNTIES)].copy()
pilot_changes["city"] = pilot_changes["county_fips"].map(PILOT_COUNTIES)
display(
    pilot_changes.pivot(index=["city", "window", "variable"], columns="scenario", values="change")
)
save_table(window_changes, "county_30year_scenario_changes.csv")
fig, axes = plt.subplots(1, 3, figsize=(15, 5), layout="constrained")
for axis, variable in zip(axes, VARIABLES, strict=True):
    arrays, ticklabels, colors = [], [], []
    for window in FUTURE_WINDOWS:
        for scenario, (color, _, label) in SCENARIO_STYLES.items():
            arrays.append(
                window_changes.loc[
                    window_changes["variable"].eq(variable)
                    & window_changes["window"].eq(window)
                    & window_changes["scenario"].eq(scenario),
                    "change",
                ]
            )
            ticklabels.append(f"{window}\n{label}")
            colors.append(color)
    boxes = axis.boxplot(
        arrays,
        tick_labels=ticklabels,
        patch_artist=True,
        showfliers=True,
        medianprops={"color": "#222222"},
        widths=0.6,
    )
    for patch, color in zip(boxes["boxes"], colors, strict=True):
        patch.set(facecolor=color, alpha=0.4)
    axis.axhline(0, color="#43484c", linewidth=0.8)
    axis.set(title=variable, ylabel="Δ tổng mưa (%)" if variable == "pcpn" else "Δ nhiệt độ (°C)")
    axis.tick_params(axis="x", labelsize=8)
fig.suptitle(
    f"Phân phối thay đổi giữa {mapped['county_fips'].nunique()} county · "
    f"baseline LOCA {BASELINE[0]}–{BASELINE[1]}"
)
show_figure(fig, "07_all_county_30year_changes.png")

## Kiểm tra

Tổng hợp kết quả và xác nhận input không đổi trước khi lưu artifacts.

In [ ]:
missing_rows = int(stations["value"].isna().sum())
latest_min = freshness["latest_numeric_date"].min().strftime("%d/%m/%Y")
latest_max = freshness["latest_numeric_date"].max().strftime("%d/%m/%Y")
late_tmax = (
    window_changes.loc[
        window_changes["variable"].eq("tmax") & window_changes["window"].eq("2070–2099")
    ]
    .groupby("scenario")["change"]
    .median()
)
summary = {
    "businesses": len(mapping),
    "mapped_us_businesses": len(mapped),
    "outside_source_businesses": int(mapping["status"].eq("outside_us_source").sum()),
    "counties": mapped["county_fips"].nunique(),
    "stations": stations["station_id"].nunique(),
    "county_rows": len(counties),
    "station_rows": len(stations),
    "station_missing_rows": missing_rows,
    "station_missing_pct": missing_rows / len(stations) * 100,
    "station_not_usable_rows": int((~stations["usable"]).sum()),
    "latest_observed_date_min": freshness["latest_numeric_date"].min().strftime("%Y-%m-%d"),
    "latest_observed_date_max": freshness["latest_numeric_date"].max().strftime("%Y-%m-%d"),
    "station_distance_median_km": float(station_distance.median()),
    "station_distance_p95_km": float(station_distance.quantile(0.95)),
    "late_century_tmax_median_change_degreeC": {
        str(key): float(value) for key, value in late_tmax.items()
    },
    "complete_month_counts": {
        str(row.variable): int(row.complete_months) for row in monthly_summary.itertuples()
    },
}
summary_text = f"""
- **{len(mapped):,} business Mỹ**, {summary["counties"]} county và {summary["stations"]} trạm;
  {summary["outside_source_businesses"]:,} business ngoài nguồn Mỹ.
- **{missing_rows:,}/{len(stations):,} dòng quan trắc thiếu
  ({summary["station_missing_pct"]:.2f}%)**;
  policy daily summary loại thêm cờ không phù hợp. Missing không trở thành mưa bằng 0.
- Dữ liệu numeric năm {latest_year} mới nhất theo station/variable nằm trong
  **{latest_min}–{latest_max}**. Năm này chưa đủ; không dùng để so tổng năm với năm hoàn chỉnh.
- Trạm cách business median **{station_distance.median():.1f} km**,
  P95 **{station_distance.quantile(0.95):.1f} km**;
  không thể diễn giải như đo weather ngay tại địa chỉ shop.
- Giữa các county (mỗi county một trọng số), median thay đổi mean daily tmax giai đoạn
  **2070–2099** so với modeled **1976–2005** là **{late_tmax["rcp45"]:.2f}°C (RCP4.5)** và
  **{late_tmax["rcp85"]:.2f}°C (RCP8.5)**. Đây là conditional model output,
  không phải forecast ngày cụ thể.
- Một tỷ lệ missing toàn scope có thể che khuất gap lớn của từng trạm/biến.
  Xem bảng pilot complete-month coverage và heatmap trước khi so sánh thành phố.
- Các bảng all-county và biểu đồ cho phép đổi pilot parameters rồi chạy lại.
  Trước khi dùng station data làm feature cần thống nhất observation windows với ngày của Yelp;
  county scenarios phù hợp khám phá biến đổi khí hậu dài hạn, giữ riêng với dữ liệu check-in.
"""

### Lưu artifacts

Xác nhận input không thay đổi trong lúc phân tích. CSV summary, PNG và manifest được lưu
dưới `data/processed/climate/`. Source notebook giữ outputs rỗng;
bản đã chạy nằm tại `data/processed/climate/02_climate_dataset_exploration.executed.ipynb`.

In [ ]:
# Confirm the inputs did not change while the notebook was running.
assert all(sha256(DATA / relative) == digest for relative, digest in INPUT_HASHES.items())
manifest = {
    "status": "complete",
    "executed_at_utc": datetime.now(UTC).isoformat(),
    "notebook": "notebooks/02_climate_dataset_exploration.ipynb",
    "input_hashes": INPUT_HASHES,
    "drive_archive": {"url": DRIVE_URL, "sha256": archive_hash},
    "parameters": {
        "pilot_counties": PILOT_COUNTIES,
        "pilot_cities": PILOT_CITIES,
        "baseline_years": BASELINE,
        "future_windows": FUTURE_WINDOWS,
    },
    "policy": {
        "usable_flags": ["blank", "T"],
        "trace_summary_mm": 0,
        "accumulations_excluded": True,
        "only_complete_calendar_months": True,
        "station_dates": "local observation windows; not UTC",
        "units": "degreeC / mm",
        "scenario_change_baseline": "same-county historical modeled weighted_mean",
        "county_distribution_weight": "equal county weights",
    },
    "summary": summary,
    "artifacts": [{"path": name, "sha256": sha256(OUTPUT / name)} for name in TABLES + FIGURES],
    "validation": {
        "unique_keys": "passed",
        "periods_and_ids": "passed",
        "input_checksums": "passed",
        "latest_dates_match_source_manifest": "passed",
        "finite_values": "passed",
        "invalid_temperature_pairs": invalid_temperature_pairs,
        "negative_precipitation": negative_precipitation,
    },
}
(OUTPUT / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + "\n")
print(f"Complete: {len(TABLES)} tables, {len(FIGURES)} figures.")
print("Manifest:", (OUTPUT / "manifest.json").relative_to(ROOT))

## Kết luận

Các kết quả dưới đây được tính từ snapshot đã chạy, kèm giới hạn dữ liệu.

In [ ]:
display(Markdown(summary_text))